In [8]:
# ubermag
import discretisedfield as df
import micromagneticmodel as mm
import oommfc as mc
# general
import numpy as np
# plotting
import ipywidgets as widgets
from IPython.display import display

In [9]:
# parameters of the sytem
Ms = 0.86e6 # saturation of magnetisation (A/m)
A = 13e-12 # exchange constant (J/m)
alpha = 0.5 # Gilbert damping constant
# define region of the system
# the nanopillar has a radius of 3nm and length of 100nm (that should be 600nm)
region = df.Region(p1=(-300e-9, -3e-9, -3e-9), p2=(300e-9, 3e-9, 3e-9))
# mesh and cellulation
mesh = df.Mesh(region=region, cell=(3e-9, 1e-9, 1e-9))

# create the system
sys_relax = mm.System(name='nanopillar_relax_05')

# funtion to define the value of the lagnetisation
def Ms_fun(point):
    '''
    Function to define the region where there is gonna be material.
    The direction of the magnetisation (m_value) is along x in this case
    '''
    # get the coordinates
    x, y, z = point
    # condition
    if (y**2 + z**2)**0.5 < 3e-9:
        return Ms
    else:
        return 0
    
# create system with above geometry and initial magnetisation
sys_relax.m = df.Field(mesh, nvdim=3, value=(1, 0, 0), norm=Ms_fun, valid="norm")

# add the energy
sys_relax.energy = mm.Exchange(A=A) + mm.Demag()
# add the dynamics
sys_relax.dynamics = mm.Damping(alpha=alpha) + mm.Precession(gamma0=mm.consts.gamma0)

In [10]:
# find the equilibrium state
md = mc.MinDriver()
md.drive(sys_relax, dirname='../../data/simulations/')

Running OOMMF (ExeOOMMFRunner)[2026-06-08T14:56:09]... (7.1 s)


In [11]:
# create the system for the dynamics
sys_dyn = mm.System(name='nanopillar_dynamics_05')

# parameters of the pulse
pulse_boundary  = -300.0e-9 + mesh.cell[0] # first cell slice
pulse_amplitude = 1e5 # A/m
pulse_duration  = 1e-12 # 1 ps
total_time = 200e-12 # 200 ps
save_dt = 0.5e-12 # save every 0.5 ps
# function for the pulse
# spatially-varying pulse field: on only where x < pulse_boundary
def H_pulse(point):
    x, y, z = point
    if x < pulse_boundary:
        return (0, pulse_amplitude, 0)   # along y
    else:
        return (0, 0, 0)

H_field = df.Field(mesh, nvdim=3, value=H_pulse)

# add the energy
sys_dyn.energy = mm.Exchange(A=A) + mm.Demag() + mm.Zeeman(H=H_field)
# add the dynamics
sys_dyn.dynamics = mm.Damping(alpha=0.05) + mm.Precession(gamma0=mm.consts.gamma0)

# initialize magnetization from the relaxed system
sys_dyn.m = df.Field(mesh, nvdim=3, value=sys_relax.m.array.copy(), norm=Ms, valid='norm')

In [12]:
# define the driver
td = mc.TimeDriver()

# phase 1: pulse on
n_pulse = round(pulse_duration/save_dt)
td.drive(sys_dyn, t=pulse_duration, n=n_pulse, dirname='../../data/simulations/', verbose=2)
# phase 2: pulse off
sys_dyn.energy = mm.Exchange(A=A) + mm.Demag() # remove Zeeman energy
remaining = total_time - pulse_duration # time for the simulation
n_rest = round(remaining/save_dt) # snapshots
td.drive(sys_dyn, t=remaining, n=n_rest, dirname='../../data/simulations/', verbose=2)

Running OOMMF (ExeOOMMFRunner):   0%|          | 0/2 files written [00:00]

Running OOMMF (ExeOOMMFRunner)[2026-06-08T14:56:17] took 7.3 s


Running OOMMF (ExeOOMMFRunner):   0%|          | 0/398 files written [00:00]

Running OOMMF (ExeOOMMFRunner)[2026-06-08T14:56:25] took 23.0 s
